# Exercise 3.1 — Collect Public Data from the Web
### *Please stay on the polite side of the internet fence*

So far, every exercise has worked with data that already existed on your computer. This one is different: we're going to reach out onto the internet, ask a public server nicely for some data, and bring it home. This is the same basic move behind stock tickers, weather widgets, currency converters, and most "live data" dashboards you've ever seen.

**Why learn this instead of just asking an AI assistant "what's the weather in Singapore"?**

Because an AI chatbot answering that question is *also* just calling an API like this one behind the scenes (or worse, guessing from memory). Once you know how to call an API directly, you can pull live data into your own spreadsheets, reports, and tools — reliably, on your own schedule, without needing to "ask" anything, and you'll immediately understand what's actually happening when any app claims to show you "real-time" data.

## What you'll practise
- **Libraries** — using someone else's code (the `requests` library) instead of reinventing it
- **APIs** — the standard way computer programs ask other computer programs for data
- **JSON** — the format almost all APIs speak

## The scenario
We'll call [Open-Meteo](https://open-meteo.com/), a free public weather API that requires no sign-up and no API key, and pull today's hourly temperature forecast for Singapore.

> **A note on politeness:** public APIs are usually free because people don't hammer them with requests. Don't call an API in a tight loop thousands of times, always check a service's terms of use for anything beyond casual/educational use, and cache results instead of re-fetching data you already have.

## Step 0: A quick word on `requests`

Python doesn't come with built-in tools to talk to web APIs comfortably — instead, the community built a **library** called `requests` that everybody uses. This is completely normal in programming: you rarely build everything from scratch, you stand on the shoulders of libraries other people have written and tested.

If you don't have it installed yet, uncomment and run the line below once.

In [1]:
# !pip install requests

import requests
print("requests version:", requests.__version__)


requests version: 2.33.1


## Step 1: Build the request URL and ask the API

In [2]:
# Open-Meteo takes latitude/longitude and gives back a forecast.
# Singapore's coordinates, roughly:
latitude = 1.3521
longitude = 103.8198

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": latitude,
    "longitude": longitude,
    "hourly": "temperature_2m,relative_humidity_2m",
    "timezone": "Asia/Singapore",
}

try:
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()   # raises an error if the request failed
    data = response.json()        # parses the JSON response into a Python dict
    print("Success! Pulled live data from the API.")
except Exception as e:
    print(f"Couldn't reach the live API ({e}).")
    print("No problem - falling back to the sample data included with this notebook.")
    import json
    with open("sample_weather_response.json") as f:
        data = json.load(f)


Couldn't reach the live API (403 Client Error: Forbidden for url: https://api.open-meteo.com/v1/forecast?latitude=1.3521&longitude=103.8198&hourly=temperature_2m%2Crelative_humidity_2m&timezone=Asia%2FSingapore).
No problem - falling back to the sample data included with this notebook.


**What just happened?**

- `requests.get(url, params=params)` sends a request to the API, with our latitude/longitude/etc attached as URL parameters. This is exactly what your browser does when you visit a website — just without the visual page.
- `.json()` converts the API's raw text response into a Python dictionary we can actually work with.
- The `try / except` block means: attempt the live call, but if anything goes wrong (no internet, API down, rate-limited), gracefully fall back to a saved sample instead of crashing. **Always plan for the network to fail** — it's not an edge case, it's a certainty eventually.

## Step 2: Explore the JSON structure

In [3]:
# JSON always maps to Python's own dict/list types, so it's easy to explore.
print("Top-level keys:", list(data.keys()))
print()
print("Hourly data keys:", list(data["hourly"].keys()))
print()
print("First 5 timestamps:", data["hourly"]["time"][:5])
print("First 5 temperatures:", data["hourly"]["temperature_2m"][:5])


Top-level keys: ['latitude', 'longitude', 'timezone', 'hourly']

Hourly data keys: ['time', 'temperature_2m', 'relative_humidity_2m']

First 5 timestamps: ['2026-07-03T00:00', '2026-07-03T01:00', '2026-07-03T02:00', '2026-07-03T03:00', '2026-07-03T04:00']
First 5 temperatures: [26.1, 25.9, 25.7, 25.6, 25.5]


**Key idea:** JSON (JavaScript Object Notation) is just nested dictionaries and lists — `{ "hourly": { "time": [...], "temperature_2m": [...] } }`. Once you can read a Python dictionary, you can already read JSON. That's why it's become the universal language of APIs: almost every programming language can turn it into its native data structures instantly.

## Step 3: Turn it into something useful — a simple table

In [4]:
times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]
humidity = data["hourly"]["relative_humidity_2m"]

print(f"{'TIME':<20}{'TEMP (°C)':<12}{'HUMIDITY (%)'}")
print("-" * 45)
for t, temp, hum in zip(times, temps, humidity):
    print(f"{t:<20}{temp:<12}{hum}")


TIME                TEMP (°C)   HUMIDITY (%)
---------------------------------------------
2026-07-03T00:00    26.1        88
2026-07-03T01:00    25.9        89
2026-07-03T02:00    25.7        90
2026-07-03T03:00    25.6        90
2026-07-03T04:00    25.5        91
2026-07-03T05:00    25.6        90
2026-07-03T06:00    25.9        88
2026-07-03T07:00    26.5        84
2026-07-03T08:00    27.3        78
2026-07-03T09:00    28.2        72
2026-07-03T10:00    29.0        67
2026-07-03T11:00    29.7        63
2026-07-03T12:00    30.2        60
2026-07-03T13:00    30.5        58
2026-07-03T14:00    30.6        57
2026-07-03T15:00    30.4        58
2026-07-03T16:00    30.0        61
2026-07-03T17:00    29.3        65
2026-07-03T18:00    28.5        70
2026-07-03T19:00    27.8        74
2026-07-03T20:00    27.3        78
2026-07-03T21:00    26.9        81
2026-07-03T22:00    26.5        84
2026-07-03T23:00    26.3        86


**New trick: `zip()`.** When you have several lists that line up item-by-item (the time at index 3 belongs with the temperature at index 3 and the humidity at index 3), `zip(times, temps, humidity)` lets you loop through all three *together*, instead of juggling three separate index variables. Very handy once you notice how often this situation comes up.

In [5]:
# Quick summary stats, since we're here
max_temp = max(temps)
min_temp = min(temps)
avg_temp = sum(temps) / len(temps)

print(f"Today's forecast for Singapore:")
print(f"  Max temperature: {max_temp}°C")
print(f"  Min temperature: {min_temp}°C")
print(f"  Average temperature: {avg_temp:.1f}°C")


Today's forecast for Singapore:
  Max temperature: 30.6°C
  Min temperature: 25.5°C
  Average temperature: 27.7°C


## 🎉 You did it

You just pulled live public data from a real API, safely handled the case where it might fail, parsed JSON, and turned it into a readable summary — the exact pipeline behind most "live data" features you use every day.

---

## 🧪 Try it yourself

1. Change the latitude/longitude to your hometown, or anywhere else in the world, and re-run.
2. Try a different free, no-key API — [Frankfurter](https://www.frankfurter.dev/) gives currency exchange rates the same way. What does its JSON structure look like?
3. Save the pulled data to a CSV file using the `csv` module from earlier exercises, so you have a permanent local record of today's forecast.

## 💡 Where this goes next
Exercise 3.2, coming up next, will take this exact idea — reading structured data and turning it into a summary — and apply it to a weekly report generator.